In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
df_raw = pd.read_csv('../data/Ship_Performance_Dataset.csv', parse_dates=['Date'])


In [3]:

# Add a unique Voyage_ID - this is the anchor key across all three systems
df_raw['Voyage_ID'] = range(1, len(df_raw) + 1)


In [4]:

# --- SOURCE 1: Vessel Operations ---
ops_cols = ['Voyage_ID', 'Date', 'Ship_Type', 'Route_Type',
            'Speed_Over_Ground_knots', 'Engine_Power_kW',
            'Distance_Traveled_nm', 'Draft_meters',
            'Weekly_Voyage_Count', 'Weather_Condition']
df_operations = df_raw[ops_cols].copy()


In [5]:

# --- SOURCE 2: Commercial & Finance ---
fin_cols = ['Voyage_ID', 'Date', 'Ship_Type',
            'Cargo_Weight_tons', 'Operational_Cost_USD',
            'Revenue_per_Voyage_USD', 'Average_Load_Percentage',
            'Seasonal_Impact_Score']
df_finance = df_raw[fin_cols].copy()


In [6]:

# --- SOURCE 3: Maintenance & Compliance ---
maint_cols = ['Voyage_ID', 'Date', 'Ship_Type',
              'Engine_Type', 'Maintenance_Status',
              'Turnaround_Time_hours', 'Efficiency_nm_per_kWh']
df_maintenance = df_raw[maint_cols].copy()


In [7]:

df_operations.to_csv('../data/source_operations.csv', index=False)
df_finance.to_csv('../data/source_finance.csv', index=False)
df_maintenance.to_csv('../data/source_maintenance.csv', index=False)


In [8]:

print("Three source files created:")
print(f"  source_operations.csv  - {len(df_operations):,} rows, {len(ops_cols)} columns")
print(f"  source_finance.csv     - {len(df_finance):,} rows, {len(fin_cols)} columns")
print(f"  source_maintenance.csv - {len(df_maintenance):,} rows, {len(maint_cols)} columns")

Three source files created:
  source_operations.csv  - 2,736 rows, 10 columns
  source_finance.csv     - 2,736 rows, 8 columns
  source_maintenance.csv - 2,736 rows, 7 columns


In [9]:
src_ops   = pd.read_csv('../data/source_operations.csv', parse_dates=['Date'])
src_fin   = pd.read_csv('../data/source_finance.csv', parse_dates=['Date'])
src_maint = pd.read_csv('../data/source_maintenance.csv', parse_dates=['Date'])


In [10]:

# Merge on Voyage_ID — unique per record, no Cartesian product risk
df_merged = src_ops.merge(src_fin, on='Voyage_ID', how='left', suffixes=('', '_fin'))
df_merged = df_merged.merge(src_maint, on='Voyage_ID', how='left', suffixes=('', '_maint'))


In [11]:

# Drop duplicate Date and Ship_Type columns brought in by the merges
df_merged.drop(columns=[c for c in df_merged.columns 
                         if c.endswith('_fin') or c.endswith('_maint')], inplace=True)


In [12]:

# --- RECONCILIATION ---
missing_before = df_merged.isnull().sum().sum()


In [13]:

cat_cols = ['Ship_Type', 'Route_Type', 'Engine_Type',
            'Maintenance_Status', 'Weather_Condition']


In [14]:

df_merged['Data_Quality_Flag'] = 'Complete'
for col in cat_cols:
    df_merged.loc[df_merged[col].isnull(), 'Data_Quality_Flag'] = 'Incomplete'


In [15]:

df_merged[cat_cols] = df_merged[cat_cols].fillna('Unknown')


In [16]:

missing_after = df_merged.isnull().sum().sum()


In [17]:

df_merged['Month'] = df_merged['Date'].dt.to_period('M')
df_merged['Month_str'] = df_merged['Date'].dt.strftime('%Y-%m')


In [18]:

df = df_merged.copy()


In [19]:

print("=== MULTI-SOURCE MERGE & RECONCILIATION REPORT ===")
print(f"\nSource 1 — Operations  : {len(src_ops):,} rows")
print(f"Source 2 — Finance     : {len(src_fin):,} rows")
print(f"Source 3 — Maintenance : {len(src_maint):,} rows")
print(f"\nMerged dataset         : {len(df):,} rows, {len(df.columns)} columns")
print(f"Missing values before reconciliation : {missing_before}")
print(f"Missing values after reconciliation  : {missing_after}")
print(f"\nData Quality Breakdown:")
print(df['Data_Quality_Flag'].value_counts().to_string())
print(f"\nDate range : {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Ship Types : {sorted([x for x in df['Ship_Type'].unique() if x != 'Unknown'])}")

=== MULTI-SOURCE MERGE & RECONCILIATION REPORT ===

Source 1 — Operations  : 2,736 rows
Source 2 — Finance     : 2,736 rows
Source 3 — Maintenance : 2,736 rows

Merged dataset         : 2,736 rows, 22 columns
Missing values before reconciliation : 680
Missing values after reconciliation  : 0

Data Quality Breakdown:
Data_Quality_Flag
Complete      2127
Incomplete     609

Date range : 2023-06-04 to 2024-06-30
Ship Types : ['Bulk Carrier', 'Container Ship', 'Fish Carrier', 'Tanker']


In [20]:
df.to_csv('../data/merged_fleet_data.csv', index=False)

In [21]:
kpi_summary = pd.DataFrame({
    'KPI': [
        'Average Speed (knots)',
        'Average Fuel Efficiency (nm/kWh)',
        'Average Operational Cost (USD)',
        'Average Revenue per Voyage (USD)',
        'Average Cargo Load (%)',
        'Average Turnaround Time (hours)',
        'Average Engine Power (kW)',
        'Total Voyages Recorded'
    ],
    'Value': [
        round(df['Speed_Over_Ground_knots'].mean(), 2),
        round(df['Efficiency_nm_per_kWh'].mean(), 4),
        round(df['Operational_Cost_USD'].mean(), 2),
        round(df['Revenue_per_Voyage_USD'].mean(), 2),
        round(df['Average_Load_Percentage'].mean(), 2),
        round(df['Turnaround_Time_hours'].mean(), 2),
        round(df['Engine_Power_kW'].mean(), 2),
        len(df)
    ]
})


In [22]:

print("=== FLEET KPI SUMMARY ===")
print(kpi_summary.to_string(index=False))

=== FLEET KPI SUMMARY ===
                             KPI       Value
           Average Speed (knots)     17.6000
Average Fuel Efficiency (nm/kWh)      0.7987
  Average Operational Cost (USD) 255143.3400
Average Revenue per Voyage (USD) 521362.0600
          Average Cargo Load (%)     75.2200
 Average Turnaround Time (hours)     41.7500
       Average Engine Power (kW)   1757.6100
          Total Voyages Recorded   2736.0000


In [23]:
from scipy import stats


In [24]:

df['Efficiency_zscore'] = np.abs(stats.zscore(df['Efficiency_nm_per_kWh']))
df['Cost_zscore'] = np.abs(stats.zscore(df['Operational_Cost_USD']))
df['Speed_zscore'] = np.abs(stats.zscore(df['Speed_Over_Ground_knots']))


In [25]:
df[['Voyage_ID', 'Date', 'Ship_Type', 'Efficiency_nm_per_kWh', 'Efficiency_zscore', 
    'Operational_Cost_USD', 'Cost_zscore', 
    'Speed_Over_Ground_knots', 'Speed_zscore']].round(4)

,Voyage_ID,Date,Ship_Type,Efficiency_nm_per_kWh,Efficiency_zscore,Operational_Cost_USD,Cost_zscore,Speed_Over_Ground_knots,Speed_zscore
0,1,2023-06-04,Container Ship,1.4552,1.6270,483832.3545,1.6235,12.5976,1.1610
1,2,2023-06-11,Fish Carrier,0.2904,1.2597,483388.0005,1.6203,10.3876,1.6736
2,3,2023-06-18,Container Ship,0.4996,0.7411,448543.4040,1.3729,20.7497,0.7299
3,4,2023-06-25,Bulk Carrier,0.7029,0.2373,261349.6054,0.0441,21.0551,0.8008
4,5,2023-07-02,Fish Carrier,1.3313,1.3201,287718.3752,0.2313,13.7428,0.8954
...,...,...,...,...,...,...,...,...,...
2731,2732,2024-06-02,Tanker,1.0003,0.4996,237975.0673,0.1219,11.6080,1.3905
2732,2733,2024-06-09,Bulk Carrier,0.6535,0.3598,21029.0217,1.6620,13.8528,0.8698
2733,2734,2024-06-16,Container Ship,0.5942,0.5068,78883.3125,1.2513,16.8137,0.1830
2734,2735,2024-06-23,Tanker,0.8957,0.2404,25241.5502,1.6321,23.1326,1.2827


In [26]:
df[['Voyage_ID', 'Date', 'Ship_Type', 'Efficiency_nm_per_kWh', 'Efficiency_zscore', 
    'Operational_Cost_USD', 'Cost_zscore', 
    'Speed_Over_Ground_knots', 'Speed_zscore']]\
    .sort_values('Efficiency_zscore', ascending=False)\
    .head(20)\
    .round(4)

,Voyage_ID,Date,Ship_Type,Efficiency_nm_per_kWh,Efficiency_zscore,Operational_Cost_USD,Cost_zscore,Speed_Over_Ground_knots,Speed_zscore
284,285,2024-06-30,Fish Carrier,1.4993,1.7362,59361.8184,1.3899,13.4575,0.9615
171,172,2023-06-04,Bulk Carrier,1.4990,1.7357,173412.3850,0.5802,13.5669,0.9362
1142,1143,2023-06-18,Tanker,1.4982,1.7336,230790.0730,0.1729,11.8980,1.3233
77,78,2023-10-22,Tanker,0.1002,1.7309,255586.3895,0.0031,21.4553,0.8936
1478,1479,2024-06-09,Tanker,1.4970,1.7307,251381.7256,0.0267,12.4342,1.1989
140,141,2023-12-03,Tanker,0.1006,1.7300,24831.5200,1.6350,11.2800,1.4666
364,365,2023-11-05,Container Ship,1.4967,1.7298,122353.6850,0.9427,14.4436,0.7328
716,717,2024-01-14,Tanker,1.4964,1.7291,390303.3385,0.9595,13.4167,0.9710
2427,2428,2024-01-21,Fish Carrier,0.1009,1.7291,126703.8247,0.9118,18.8697,0.2939
407,408,2023-07-30,Bulk Carrier,0.1015,1.7278,28867.7844,1.6063,11.3743,1.4447


In [27]:
df[['Efficiency_zscore', 'Cost_zscore', 'Speed_zscore']].describe().round(4)

,Efficiency_zscore,Cost_zscore,Speed_zscore
count,2736.0000,2736.0000,2736.0000
mean,0.8661,0.8686,0.8621
std,0.4999,0.4956,0.5069
min,0.0003,0.0000,0.0000
25%,0.4428,0.4463,0.4142
50%,0.8545,0.8898,0.8538
75%,1.2975,1.2882,1.2910
max,1.7362,1.7396,1.7613


In [28]:

# Lowered threshold to 1.5 — standard for operational KPI monitoring
# where early warning matters more than only catching extreme outliers
THRESHOLD = 1.5

df['Is_Anomaly'] = (
    (df['Efficiency_zscore'] > THRESHOLD) |
    (df['Cost_zscore'] > THRESHOLD) |
    (df['Speed_zscore'] > THRESHOLD)
)


In [29]:

# Classify anomaly type for richer reporting
def classify_anomaly(row):
    flags = []
    if row['Efficiency_zscore'] > THRESHOLD:
        flags.append('Low Efficiency')
    if row['Cost_zscore'] > THRESHOLD:
        flags.append('High Cost')
    if row['Speed_zscore'] > THRESHOLD:
        flags.append('Speed Deviation')
    return ', '.join(flags) if flags else 'Normal'


In [31]:

df['Anomaly_Type'] = df.apply(classify_anomaly, axis=1)


In [32]:
anomalies = df[df['Is_Anomaly'] == True].copy()


In [33]:

print("=== ANOMALY DETECTION RESULTS ===")
print(f"\nTotal records    : {len(df):,}")
print(f"Anomalies found  : {len(anomalies):,}")
print(f"Anomaly rate     : {round(len(anomalies)/len(df)*100, 2)}%")
print(f"\nAnomalies by Ship Type:")
print(anomalies['Ship_Type'].value_counts().to_string())
print(f"\nAnomalies by Maintenance Status:")
print(anomalies['Maintenance_Status'].value_counts().to_string())
print(f"\nAnomalies by Route Type:")
print(anomalies['Route_Type'].value_counts().to_string())
print(f"\nAnomaly Type Breakdown:")
print(anomalies['Anomaly_Type'].value_counts().to_string())

=== ANOMALY DETECTION RESULTS ===

Total records    : 2,736
Anomalies found  : 975
Anomaly rate     : 35.64%

Anomalies by Ship Type:
Ship_Type
Tanker            248
Bulk Carrier      238
Fish Carrier      234
Container Ship    213
Unknown            42

Anomalies by Maintenance Status:
Maintenance_Status
Critical    321
Fair        305
Good        304
Unknown      45

Anomalies by Route Type:
Route_Type
Transoceanic    241
Long-haul       239
Short-haul      234
Coastal         218
Unknown          43

Anomaly Type Breakdown:
Anomaly_Type
Speed Deviation                               302
Low Efficiency                                282
High Cost                                     256
Low Efficiency, Speed Deviation                45
Low Efficiency, High Cost                      43
High Cost, Speed Deviation                     40
Low Efficiency, High Cost, Speed Deviation      7


In [34]:
avg_cost    = df[df['Is_Anomaly'] == False]['Operational_Cost_USD'].mean()
avg_revenue = df[df['Is_Anomaly'] == False]['Revenue_per_Voyage_USD'].mean()
avg_efficiency = df[df['Is_Anomaly'] == False]['Efficiency_nm_per_kWh'].mean()


In [35]:

anomalies = anomalies.copy()
anomalies['Excess_Cost_USD']    = anomalies['Operational_Cost_USD'] - avg_cost
anomalies['Lost_Revenue_USD']   = (avg_revenue - anomalies['Revenue_per_Voyage_USD']).clip(lower=0)
anomalies['Efficiency_Gap']     = avg_efficiency - anomalies['Efficiency_nm_per_kWh']


In [36]:

total_excess_cost    = anomalies['Excess_Cost_USD'].sum()
total_lost_revenue   = anomalies['Lost_Revenue_USD'].sum()
total_impact         = total_excess_cost + total_lost_revenue


In [37]:

# Annualise the impact
dataset_months = 13
annual_factor  = 12 / dataset_months
annual_impact  = total_impact * annual_factor


In [39]:
print("=== STEP BY STEP IMPACT BREAKDOWN ===")

print(f"\n--- Per Voyage Gap (sample) ---")
print(anomalies[['Voyage_ID', 'Ship_Type', 'Operational_Cost_USD', 
                  'Excess_Cost_USD', 'Revenue_per_Voyage_USD', 
                  'Lost_Revenue_USD', 'Efficiency_Gap']].head(10).round(2).to_string())

print(f"\n--- Summing Across All 975 Anomalous Voyages ---")
print(f"  Total excess cost     : ${total_excess_cost:>15,.2f}  (sum of Excess_Cost_USD column)")
print(f"  Total lost revenue    : ${total_lost_revenue:>15,.2f}  (sum of Lost_Revenue_USD column)")
print(f"  Combined impact       : ${total_impact:>15,.2f}  (excess cost + lost revenue)")

print(f"\n--- Annualising ---")
print(f"  Dataset spans         : {dataset_months} months")
print(f"  Annual factor         : 12 / {dataset_months} = {annual_factor:.4f}")
print(f"  Combined impact       : ${total_impact:>15,.2f}")
print(f"  x annual factor       : x {annual_factor:.4f}")
print(f"  = Annual impact       : ${annual_impact:>15,.2f}")

=== STEP BY STEP IMPACT BREAKDOWN ===

--- Per Voyage Gap (sample) ---
    Voyage_ID       Ship_Type  Operational_Cost_USD  Excess_Cost_USD  Revenue_per_Voyage_USD  Lost_Revenue_USD  Efficiency_Gap
0           1  Container Ship             483832.35        229766.99               292183.27         223498.71           -0.66
1           2    Fish Carrier             483388.00        229322.63               883765.79              0.00            0.51
11         12          Tanker             481866.26        227800.89               738063.17              0.00           -0.43
12         13  Container Ship             481846.66        227781.29               738287.28              0.00            0.22
22         23         Unknown              14878.15       -239187.21               902159.37              0.00           -0.13
23         24  Container Ship             471255.43        217190.06               880435.93              0.00            0.62
29         30    Fish Carrier           

In [38]:

print("=== ROI & COST IMPACT ANALYSIS ===")
print(f"\nFleet benchmark — normal voyages:")
print(f"  Avg operational cost  : ${avg_cost:>12,.2f}")
print(f"  Avg revenue           : ${avg_revenue:>12,.2f}")
print(f"  Avg efficiency        : {avg_efficiency:.4f} nm/kWh")
print(f"\nImpact from {len(anomalies)} anomalous voyages:")
print(f"  Total excess cost     : ${total_excess_cost:>12,.2f}")
print(f"  Total lost revenue    : ${total_lost_revenue:>12,.2f}")
print(f"  Combined impact       : ${total_impact:>12,.2f}")
print(f"\n>>> Estimated annual financial impact if unresolved:")
print(f">>> ${annual_impact:>12,.2f} per year")
print(f"\nResolving these anomalies represents a potential")
print(f"annual saving of ${annual_impact:,.2f} for the fleet.")

=== ROI & COST IMPACT ANALYSIS ===

Fleet benchmark — normal voyages:
  Avg operational cost  : $  254,065.37
  Avg revenue           : $  515,681.98
  Avg efficiency        : 0.7984 nm/kWh

Impact from 975 anomalous voyages:
  Total excess cost     : $2,949,346.92
  Total lost revenue    : $109,585,577.76
  Combined impact       : $112,534,924.68

>>> Estimated annual financial impact if unresolved:
>>> $103,878,392.01 per year

Resolving these anomalies represents a potential
annual saving of $103,878,392.01 for the fleet.


In [40]:
# Save anomaly records for reporting
anomaly_export = anomalies[[
    'Voyage_ID', 'Date', 'Ship_Type', 'Route_Type',
    'Maintenance_Status', 'Anomaly_Type',
    'Speed_Over_Ground_knots', 'Efficiency_nm_per_kWh',
    'Operational_Cost_USD', 'Revenue_per_Voyage_USD',
    'Excess_Cost_USD', 'Lost_Revenue_USD'
]].copy()


In [41]:

anomaly_export['Date'] = anomaly_export['Date'].dt.strftime('%Y-%m-%d')
anomaly_export = anomaly_export.round(2)
anomaly_export.to_csv('../outputs/anomaly_summary.csv', index=False)


In [42]:

print(f"Anomaly summary exported: {len(anomaly_export)} records")
print(f"Saved to: outputs/anomaly_summary.csv")

Anomaly summary exported: 975 records
Saved to: outputs/anomaly_summary.csv


In [43]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px

# ── Prepare data ──────────────────────────────────────────────────────────────

# 1. Monthly trend — avg cost & efficiency over time
monthly = (df.groupby('Month_str')
             .agg(Avg_Cost=('Operational_Cost_USD','mean'),
                  Avg_Efficiency=('Efficiency_nm_per_kWh','mean'),
                  Avg_Speed=('Speed_Over_Ground_knots','mean'),
                  Voyage_Count=('Voyage_ID','count'))
             .reset_index()
             .sort_values('Month_str'))

# 2. Anomaly rate by ship type
anomaly_rate = (df.groupby('Ship_Type')
                  .apply(lambda x: round(x['Is_Anomaly'].sum()/len(x)*100, 2))
                  .reset_index()
                  .rename(columns={0:'Anomaly_Rate'})
                  .query("Ship_Type != 'Unknown'"))

# 3. Cost vs Revenue by route type
route_perf = (df[df['Route_Type'] != 'Unknown']
              .groupby('Route_Type')
              .agg(Avg_Cost=('Operational_Cost_USD','mean'),
                   Avg_Revenue=('Revenue_per_Voyage_USD','mean'))
              .reset_index())

# 4. Maintenance status vs avg operational cost
maint_cost = (df[df['Maintenance_Status'] != 'Unknown']
              .groupby('Maintenance_Status')
              .agg(Avg_Cost=('Operational_Cost_USD','mean'),
                   Avg_Efficiency=('Efficiency_nm_per_kWh','mean'))
              .reset_index()
              .sort_values('Avg_Cost', ascending=False))

# 5. Anomaly type breakdown
anomaly_types = (anomalies['Anomaly_Type']
                 .value_counts()
                 .reset_index()
                 .rename(columns={'index':'Type','Anomaly_Type':'Count'}))
anomaly_types.columns = ['Anomaly_Type','Count']

# 6. ROI summary values
total_excess   = anomalies['Excess_Cost_USD'].sum()
total_lost_rev = anomalies['Lost_Revenue_USD'].sum()
annual_impact  = (total_excess + total_lost_rev) * (12/13)

# ── Build dashboard ───────────────────────────────────────────────────────────

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        'Monthly Avg Operational Cost (USD)',
        'Anomaly Rate by Ship Type (%)',
        'Avg Cost vs Revenue by Route Type',
        'Maintenance Status vs Avg Cost',
        'Anomaly Type Breakdown',
        'ROI Impact Summary'
    ),
    specs=[
        [{"type": "scatter"}, {"type": "bar"}],
        [{"type": "bar"},     {"type": "bar"}],
        [{"type": "pie"},     {"type": "bar"}]
    ],
    vertical_spacing=0.14,
    horizontal_spacing=0.10
)

# Hapag-Lloyd brand colours
HL_ORANGE = '#FF6600'
HL_BLUE   = '#003591'
HL_GREY   = '#8C8C8C'
HL_GREEN  = '#00A86B'
HL_RED    = '#D62728'

# ── Plot 1: Monthly cost trend ────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=monthly['Month_str'], y=monthly['Avg_Cost'],
    mode='lines+markers',
    line=dict(color=HL_ORANGE, width=2.5),
    marker=dict(size=6),
    name='Avg Cost'
), row=1, col=1)

# ── Plot 2: Anomaly rate by ship type ─────────────────────────────────────────
fig.add_trace(go.Bar(
    x=anomaly_rate['Ship_Type'],
    y=anomaly_rate['Anomaly_Rate'],
    marker_color=HL_RED,
    text=anomaly_rate['Anomaly_Rate'].astype(str) + '%',
    textposition='outside',
    name='Anomaly Rate'
), row=1, col=2)

# ── Plot 3: Cost vs Revenue by route ─────────────────────────────────────────
fig.add_trace(go.Bar(
    x=route_perf['Route_Type'], y=route_perf['Avg_Cost'],
    name='Avg Cost', marker_color=HL_RED
), row=2, col=1)
fig.add_trace(go.Bar(
    x=route_perf['Route_Type'], y=route_perf['Avg_Revenue'],
    name='Avg Revenue', marker_color=HL_GREEN
), row=2, col=1)

# ── Plot 4: Maintenance vs cost ───────────────────────────────────────────────
fig.add_trace(go.Bar(
    x=maint_cost['Maintenance_Status'],
    y=maint_cost['Avg_Cost'],
    marker_color=[HL_RED if x == 'Critical' else 
                  HL_ORANGE if x == 'Fair' else 
                  HL_GREEN for x in maint_cost['Maintenance_Status']],
    text=maint_cost['Avg_Cost'].apply(lambda x: f'${x:,.0f}'),
    textposition='outside',
    name='Maintenance Cost'
), row=2, col=2)

# ── Plot 5: Anomaly type pie ──────────────────────────────────────────────────
fig.add_trace(go.Pie(
    labels=anomaly_types['Anomaly_Type'],
    values=anomaly_types['Count'],
    hole=0.4,
    marker_colors=[HL_ORANGE, HL_RED, HL_BLUE, 
                   HL_GREY, HL_GREEN, '#9467BD', '#8C564B'],
    textinfo='percent+label'
), row=3, col=1)

# ── Plot 6: ROI bar ───────────────────────────────────────────────────────────
roi_labels = ['Excess Operational Cost', 'Lost Revenue', 'Total Annual Impact']
roi_values = [total_excess, total_lost_rev, annual_impact]
roi_colors = [HL_ORANGE, HL_RED, HL_BLUE]

fig.add_trace(go.Bar(
    x=roi_labels, y=roi_values,
    marker_color=roi_colors,
    text=[f'${v/1e6:.1f}M' for v in roi_values],
    textposition='outside',
    name='ROI Impact'
), row=3, col=2)

# ── Layout ────────────────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text='Hapag-Lloyd Fleet Performance KPI Dashboard',
        font=dict(size=22, color=HL_BLUE),
        x=0.5
    ),
    height=1100,
    showlegend=False,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=11, color='#333333')
)

# Clean axis styling
fig.update_xaxes(showgrid=False, linecolor='#CCCCCC')
fig.update_yaxes(showgrid=True, gridcolor='#F0F0F0', linecolor='#CCCCCC')

# Rotate x-axis labels on monthly trend
fig.update_xaxes(tickangle=45, row=1, col=1)

# Save as interactive HTML
fig.write_html('../dashboard/fleet_dashboard.html')
print("Dashboard saved to: dashboard/fleet_dashboard.html")
print("Open that file in your browser to view the interactive dashboard.")

Dashboard saved to: dashboard/fleet_dashboard.html
Open that file in your browser to view the interactive dashboard.


In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import (PatternFill, Font, Alignment, 
                              Border, Side, numbers)
from openpyxl.utils import get_column_letter
from openpyxl.chart import BarChart, Reference
import openpyxl

wb = openpyxl.Workbook()

# ── Colour palette ────────────────────────────────────────────────────────────
HL_BLUE   = '003591'
HL_ORANGE = 'FF6600'
HL_WHITE  = 'FFFFFF'
HL_LGREY  = 'F5F5F5'
HL_DGREY  = 'CCCCCC'
HL_RED    = 'D62728'
HL_GREEN  = '00A86B'

def header_fill(colour): 
    return PatternFill('solid', fgColor=colour)

def bold_white(size=11): 
    return Font(bold=True, color=HL_WHITE, size=size, name='Arial')

def bold_dark(size=11):  
    return Font(bold=True, color='333333', size=size, name='Arial')

def normal(size=10):     
    return Font(size=size, name='Arial', color='333333')

def border():
    s = Side(style='thin', color=HL_DGREY)
    return Border(left=s, right=s, top=s, bottom=s)

def set_col_width(ws, col, width):
    ws.column_dimensions[get_column_letter(col)].width = width

# ══════════════════════════════════════════════════════════════════════════════
# SHEET 1 — Executive Summary
# ══════════════════════════════════════════════════════════════════════════════
ws1 = wb.active
ws1.title = 'Executive Summary'

# Title block
ws1.merge_cells('A1:F1')
ws1['A1'] = 'Hapag-Lloyd Fleet Performance Report'
ws1['A1'].font = Font(bold=True, size=16, color=HL_WHITE, name='Arial')
ws1['A1'].fill = header_fill(HL_BLUE)
ws1['A1'].alignment = Alignment(horizontal='center', vertical='center')
ws1.row_dimensions[1].height = 36

ws1.merge_cells('A2:F2')
ws1['A2'] = f'Analysis Period: June 2023 – June 2024  |  Total Voyages: {len(df):,}  |  Generated: March 2026'
ws1['A2'].font = Font(size=10, color=HL_WHITE, name='Arial')
ws1['A2'].fill = header_fill(HL_ORANGE)
ws1['A2'].alignment = Alignment(horizontal='center', vertical='center')
ws1.row_dimensions[2].height = 22

# KPI header
ws1.merge_cells('A4:F4')
ws1['A4'] = 'KEY PERFORMANCE INDICATORS'
ws1['A4'].font = bold_dark(11)
ws1['A4'].fill = header_fill('E8EDF5')
ws1['A4'].alignment = Alignment(horizontal='left', vertical='center')
ws1.row_dimensions[4].height = 20

# KPI table headers
kpi_headers = ['KPI', 'Value', 'Unit', 'Benchmark', 'Status', 'Notes']
for col, h in enumerate(kpi_headers, 1):
    c = ws1.cell(row=5, column=col, value=h)
    c.font = bold_white()
    c.fill = header_fill(HL_BLUE)
    c.alignment = Alignment(horizontal='center', vertical='center')
    c.border = border()
ws1.row_dimensions[5].height = 20

# KPI data rows
kpi_rows = [
    ('Average Speed',           round(df['Speed_Over_Ground_knots'].mean(),2),  'knots',   '15–20',    'Normal',   'Within expected operating range'),
    ('Fuel Efficiency',         round(df['Efficiency_nm_per_kWh'].mean(),4),    'nm/kWh',  '>0.75',    'Normal',   'Above minimum efficiency threshold'),
    ('Avg Operational Cost',    round(df['Operational_Cost_USD'].mean(),0),     'USD',     '<300,000', 'Normal',   'Within budget parameters'),
    ('Avg Revenue per Voyage',  round(df['Revenue_per_Voyage_USD'].mean(),0),   'USD',     '>400,000', 'Normal',   'Exceeds revenue target'),
    ('Avg Cargo Load',          round(df['Average_Load_Percentage'].mean(),2),  '%',       '>70%',     'Normal',   'Good capacity utilisation'),
    ('Avg Turnaround Time',     round(df['Turnaround_Time_hours'].mean(),2),    'hours',   '<50',      'Normal',   'Acceptable port turnaround'),
    ('Anomaly Rate',            round(len(anomalies)/len(df)*100,2),            '%',       '<15%',     'Warning',  'Elevated - review recommended'),
    ('Total Voyages Analysed',  len(df),                                        'voyages', '—',        '—',        '13-month analysis window'),
]

status_colours = {'Normal': HL_GREEN, 'Warning': HL_ORANGE, '—': HL_LGREY}

for r, row_data in enumerate(kpi_rows, 6):
    for c, val in enumerate(row_data, 1):
        cell = ws1.cell(row=r, column=c, value=val)
        cell.font = normal()
        cell.border = border()
        cell.alignment = Alignment(horizontal='center' if c != 1 else 'left',
                                   vertical='center')
        if c == 5:  # Status column
            colour = status_colours.get(val, HL_LGREY)
            cell.fill = PatternFill('solid', fgColor=colour)
            cell.font = Font(bold=True, color=HL_WHITE, size=10, name='Arial')
        if r % 2 == 0 and c != 5:
            cell.fill = PatternFill('solid', fgColor=HL_LGREY)
    ws1.row_dimensions[r].height = 18

# ROI Impact block
ws1.merge_cells('A15:F15')
ws1['A15'] = 'ROI & FINANCIAL IMPACT SUMMARY'
ws1['A15'].font = bold_dark(11)
ws1['A15'].fill = header_fill('E8EDF5')
ws1['A15'].alignment = Alignment(horizontal='left', vertical='center')
ws1.row_dimensions[15].height = 20

roi_data = [
    ('Anomalous Voyages Detected',   len(anomalies),          'voyages',  f'{round(len(anomalies)/len(df)*100,1)}% of total fleet activity'),
    ('Excess Operational Cost',      round(total_excess,0),   'USD',      'Cost above fleet benchmark for anomalous voyages'),
    ('Lost Revenue',                 round(total_lost_rev,0), 'USD',      'Revenue below fleet benchmark for anomalous voyages'),
    ('Combined Period Impact',       round(total_excess+total_lost_rev,0), 'USD', 'Total financial impact over 13-month period'),
    ('Estimated Annual Impact',      round(annual_impact,0),  'USD/year', 'Annualised saving if anomalies are fully resolved'),
]

roi_headers = ['Metric', 'Value', 'Unit', 'Description']
for col, h in enumerate(roi_headers, 1):
    c = ws1.cell(row=16, column=col, value=h)
    c.font = bold_white()
    c.fill = header_fill(HL_ORANGE)
    c.alignment = Alignment(horizontal='center', vertical='center')
    c.border = border()

for r, row_data in enumerate(roi_data, 17):
    for c, val in enumerate(row_data, 1):
        cell = ws1.cell(row=r, column=c, value=val)
        cell.font = normal()
        cell.border = border()
        cell.alignment = Alignment(horizontal='center' if c != 1 else 'left',
                                   vertical='center')
        if r % 2 == 0:
            cell.fill = PatternFill('solid', fgColor=HL_LGREY)
    ws1.row_dimensions[r].height = 18

# Column widths
for col, width in zip(range(1,7), [30, 18, 12, 16, 12, 45]):
    set_col_width(ws1, col, width)

# ══════════════════════════════════════════════════════════════════════════════
# SHEET 2 — Monthly Trends
# ══════════════════════════════════════════════════════════════════════════════
ws2 = wb.create_sheet('Monthly Trends')

ws2.merge_cells('A1:E1')
ws2['A1'] = 'Monthly Fleet Performance Trends'
ws2['A1'].font = Font(bold=True, size=14, color=HL_WHITE, name='Arial')
ws2['A1'].fill = header_fill(HL_BLUE)
ws2['A1'].alignment = Alignment(horizontal='center', vertical='center')
ws2.row_dimensions[1].height = 30

headers = ['Month', 'Avg Cost (USD)', 'Avg Efficiency (nm/kWh)', 
           'Avg Speed (knots)', 'Voyage Count']
for col, h in enumerate(headers, 1):
    c = ws2.cell(row=2, column=col, value=h)
    c.font = bold_white(10)
    c.fill = header_fill(HL_ORANGE)
    c.alignment = Alignment(horizontal='center', vertical='center')
    c.border = border()

for r, row in enumerate(monthly.itertuples(), 3):
    vals = [row.Month_str, round(row.Avg_Cost,0), 
            round(row.Avg_Efficiency,4),
            round(row.Avg_Speed,2), row.Voyage_Count]
    for c, val in enumerate(vals, 1):
        cell = ws2.cell(row=r, column=c, value=val)
        cell.font = normal()
        cell.border = border()
        cell.alignment = Alignment(horizontal='center', vertical='center')
        if r % 2 == 0:
            cell.fill = PatternFill('solid', fgColor=HL_LGREY)
    ws2.row_dimensions[r].height = 16

for col, width in zip(range(1,6), [14, 18, 24, 18, 14]):
    set_col_width(ws2, col, width)

# ══════════════════════════════════════════════════════════════════════════════
# SHEET 3 — Anomaly Report
# ══════════════════════════════════════════════════════════════════════════════
ws3 = wb.create_sheet('Anomaly Report')

ws3.merge_cells('A1:F1')
ws3['A1'] = f'Anomaly Detection Report  |  {len(anomalies)} Flagged Voyages  |  Anomaly Rate: {round(len(anomalies)/len(df)*100,2)}%'
ws3['A1'].font = Font(bold=True, size=13, color=HL_WHITE, name='Arial')
ws3['A1'].fill = header_fill(HL_RED)
ws3['A1'].alignment = Alignment(horizontal='center', vertical='center')
ws3.row_dimensions[1].height = 28

anom_headers = ['Date', 'Ship Type', 'Route', 'Maintenance',
                'Anomaly Type', 'Operational Cost (USD)',
                'Revenue (USD)', 'Excess Cost (USD)', 'Lost Revenue (USD)']
for col, h in enumerate(anom_headers, 1):
    c = ws3.cell(row=2, column=col, value=h)
    c.font = bold_white(10)
    c.fill = header_fill(HL_BLUE)
    c.alignment = Alignment(horizontal='center', vertical='center')
    c.border = border()

anom_display = anomalies[[
    'Date','Ship_Type','Route_Type','Maintenance_Status',
    'Anomaly_Type','Operational_Cost_USD','Revenue_per_Voyage_USD',
    'Excess_Cost_USD','Lost_Revenue_USD'
]].copy()
anom_display['Date'] = pd.to_datetime(anom_display['Date']).dt.strftime('%Y-%m-%d')

for r, row in enumerate(anom_display.itertuples(index=False), 3):
    for c, val in enumerate(row, 1):
        cell = ws3.cell(row=r, column=c, value=val)
        cell.font = normal(9)
        cell.border = border()
        cell.alignment = Alignment(horizontal='center', vertical='center')
        if r % 2 == 0:
            cell.fill = PatternFill('solid', fgColor=HL_LGREY)
    ws3.row_dimensions[r].height = 15

for col, width in zip(range(1,10), [13,16,14,14,32,22,18,18,18]):
    set_col_width(ws3, col, width)

# ══════════════════════════════════════════════════════════════════════════════
# SHEET 4 — Route & Ship Type Performance
# ══════════════════════════════════════════════════════════════════════════════
ws4 = wb.create_sheet('Route & Ship Performance')

ws4.merge_cells('A1:F1')
ws4['A1'] = 'Performance Breakdown by Route Type and Ship Type'
ws4['A1'].font = Font(bold=True, size=13, color=HL_WHITE, name='Arial')
ws4['A1'].fill = header_fill(HL_BLUE)
ws4['A1'].alignment = Alignment(horizontal='center', vertical='center')
ws4.row_dimensions[1].height = 28

# Route performance table
ws4['A3'] = 'Performance by Route Type'
ws4['A3'].font = bold_dark(11)
ws4['A3'].fill = PatternFill('solid', fgColor='E8EDF5')

route_headers = ['Route Type','Avg Cost (USD)','Avg Revenue (USD)',
                 'Avg Efficiency','Voyage Count','Anomaly Rate (%)']
for col, h in enumerate(route_headers, 1):
    c = ws4.cell(row=4, column=col, value=h)
    c.font = bold_white(10)
    c.fill = header_fill(HL_ORANGE)
    c.alignment = Alignment(horizontal='center')
    c.border = border()

route_full = (df[df['Route_Type'] != 'Unknown']
              .groupby('Route_Type')
              .agg(Avg_Cost=('Operational_Cost_USD','mean'),
                   Avg_Revenue=('Revenue_per_Voyage_USD','mean'),
                   Avg_Eff=('Efficiency_nm_per_kWh','mean'),
                   Count=('Voyage_ID','count'),
                   Anomalies=('Is_Anomaly','sum'))
              .reset_index())
route_full['Anomaly_Rate'] = (route_full['Anomalies']/route_full['Count']*100).round(2)

for r, row in enumerate(route_full.itertuples(), 5):
    vals = [row.Route_Type, round(row.Avg_Cost,0), round(row.Avg_Revenue,0),
            round(row.Avg_Eff,4), row.Count, row.Anomaly_Rate]
    for c, val in enumerate(vals, 1):
        cell = ws4.cell(row=r, column=c, value=val)
        cell.font = normal()
        cell.border = border()
        cell.alignment = Alignment(horizontal='center')
        if r % 2 == 0:
            cell.fill = PatternFill('solid', fgColor=HL_LGREY)

# Ship type performance table
ws4['A11'] = 'Performance by Ship Type'
ws4['A11'].font = bold_dark(11)
ws4['A11'].fill = PatternFill('solid', fgColor='E8EDF5')

ship_headers = ['Ship Type','Avg Cost (USD)','Avg Revenue (USD)',
                'Avg Efficiency','Voyage Count','Anomaly Rate (%)']
for col, h in enumerate(ship_headers, 1):
    c = ws4.cell(row=12, column=col, value=h)
    c.font = bold_white(10)
    c.fill = header_fill(HL_BLUE)
    c.alignment = Alignment(horizontal='center')
    c.border = border()

ship_full = (df[df['Ship_Type'] != 'Unknown']
             .groupby('Ship_Type')
             .agg(Avg_Cost=('Operational_Cost_USD','mean'),
                  Avg_Revenue=('Revenue_per_Voyage_USD','mean'),
                  Avg_Eff=('Efficiency_nm_per_kWh','mean'),
                  Count=('Voyage_ID','count'),
                  Anomalies=('Is_Anomaly','sum'))
             .reset_index())
ship_full['Anomaly_Rate'] = (ship_full['Anomalies']/ship_full['Count']*100).round(2)

for r, row in enumerate(ship_full.itertuples(), 13):
    vals = [row.Ship_Type, round(row.Avg_Cost,0), round(row.Avg_Revenue,0),
            round(row.Avg_Eff,4), row.Count, row.Anomaly_Rate]
    for c, val in enumerate(vals, 1):
        cell = ws4.cell(row=r, column=c, value=val)
        cell.font = normal()
        cell.border = border()
        cell.alignment = Alignment(horizontal='center')
        if r % 2 == 0:
            cell.fill = PatternFill('solid', fgColor=HL_LGREY)

for col, width in zip(range(1,7), [18,18,18,16,14,16]):
    set_col_width(ws4, col, width)

# ── Save workbook ─────────────────────────────────────────────────────────────
output_path = '../outputs/fleet_kpi_report.xlsx'
wb.save(output_path)
print(f"Excel report saved to: {output_path}")
print(f"Sheets created: Executive Summary | Monthly Trends | Anomaly Report | Route & Ship Performance")

Excel report saved to: ../outputs/fleet_kpi_report.xlsx
Sheets created: Executive Summary | Monthly Trends | Anomaly Report | Route & Ship Performance
